# 1. Intro

Retrieval-Augmented Generation (RAG) combines two ideas: retrieve the most relevant context from your own data, then ask an LLM to answer using that context. In this notebook we build a simple RAG system over support tickets using pandas, NumPy, Parquet, and OpenAI.


In [ ]:
# 2. Load & explore tickets
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.ingest import load_tickets

tickets_path = project_root / "data" / "support_tickets.csv"
tickets_df = load_tickets(tickets_path)
tickets_df.head(), tickets_df["category"].value_counts()


In [ ]:
# 3. What are embeddings? Optional 2D visualization with t-SNE
import numpy as np
import matplotlib.pyplot as plt

from src.ingest import prepare_chunks
from src.llm import get_client, get_embedding

sample_chunks = prepare_chunks(tickets_df.head(12))
client = get_client()
sample_vectors = np.array([get_embedding(chunk["text"], client) for chunk in sample_chunks])

try:
    from sklearn.manifold import TSNE
    coords = TSNE(n_components=2, perplexity=5, random_state=42).fit_transform(sample_vectors)
    plt.figure(figsize=(8, 5))
    for idx, chunk in enumerate(sample_chunks):
        plt.scatter(coords[idx, 0], coords[idx, 1])
        plt.text(coords[idx, 0] + 0.2, coords[idx, 1] + 0.2, chunk["category"], fontsize=8)
    plt.title("Similar ticket meanings end up near each other in vector space")
    plt.show()
except Exception as exc:
    print("t-SNE visualization skipped:", exc)


In [ ]:
# 4. Build the index
from src.embed import embed_chunks

chunks = prepare_chunks(tickets_df)
output_path = project_root / "output" / "embeddings.parquet"
embed_df = embed_chunks(chunks, client, output_path)
embed_df.head()


In [ ]:
# 5. Retrieve top-3 tickets for a sample question
from src.embed import load_embeddings
from src.retrieve import retrieve_top_k

embeddings_df = load_embeddings(output_path)
question = "My reset link expired right away. How can I get back into my account?"
question_embedding = np.array(get_embedding(question, client), dtype=float)
top_hits = retrieve_top_k(question_embedding, embeddings_df, k=3)
top_hits


In [ ]:
# 6. Generate answer: full RAG query
from src.query import answer_question

result = answer_question(question, embeddings_df, client, k=3)
print(result["answer"])
print("Sources:", result["sources"])


In [ ]:
# 7. Try different questions and watch retrieval change
questions = [
    "Why is my refund not back on my card yet?",
    "Can you unlock my account after failed logins?",
    "My package says delivered but I cannot find it.",
]

for q in questions:
    result = answer_question(q, embeddings_df, client, k=3)
    print(f"\nQuestion: {q}")
    print("Sources:", result["sources"])


# 8. Adapting to Databricks / production

- Replace local Parquet storage with Delta Lake tables for chunk metadata and embeddings.
- Schedule the indexing step as a Databricks Workflow so new tickets are embedded automatically.
- Move retrieval to Databricks Vector Search or another vector index when the dataset grows beyond a simple in-memory scan.
- Add evaluation, monitoring, and source citation logging before exposing the system to business users.
